# HypeGuard — Notebook 01: Offline EDA on Prepared Dataset
Uses existing CSV artifacts from `../data/` and does not call live market/news APIs.

Run this notebook first, then Notebook 02 (training), then Notebook 03 (validation).

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — IMPORTS + OFFLINE DATA LOADING
# Uses existing CSV artifacts only (no live API calls).
# ═══════════════════════════════════════════════════════════

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'text.color':       'white',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'axes.edgecolor':   '#444',
    'grid.color':       '#333',
    'font.family':      'monospace',
})

RED    = '#ef4444'
GREEN  = '#22c55e'
BLUE   = '#3b82f6'
AMBER  = '#f59e0b'
PURPLE = '#a855f7'
GRAY   = '#6b7280'

LABEL_COLORS = {'ORGANIC': GREEN, 'HYPE': RED, 'INSTITUTIONAL': BLUE, 'NEUTRAL': GRAY}

DATA_PATH   = '../data/training_data.csv'
TRAIN_PATH  = '../data/training_train.csv'
VAL_PATH    = '../data/training_val.csv'
REPORT_PATH = '../data/training_report.json'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
df_train = pd.read_csv(TRAIN_PATH) if os.path.exists(TRAIN_PATH) else pd.DataFrame()
df_val = pd.read_csv(VAL_PATH) if os.path.exists(VAL_PATH) else pd.DataFrame()
report = {}
if os.path.exists(REPORT_PATH):
    with open(REPORT_PATH, 'r', encoding='utf-8') as f:
        report = json.load(f)

FEATURE_ORDER = [
    'rvol', 'volume_zscore', 'vol_price_divergence', 'vol_spike_days', 'vol_trend_slope_norm',
    'log_return_1d', 'price_vs_sma20', 'rsi_14', 'bb_width', 'gap_open', 'range_expansion',
    'buzz_density', 'extreme_language_ratio', 'moderate_hype_ratio',
    'bearish_ratio', 'source_diversity', 'headline_similarity',
    'catalyst_flag', 'hype_without_catalyst', 'news_volume_sync', 'silent_spike',
]

FEATURE_ORDER = [c for c in FEATURE_ORDER if c in df.columns]
print('✓ Offline dataset loaded')
print(f'  data rows      : {len(df)}')
print(f'  train rows     : {len(df_train)}')
print(f'  val rows       : {len(df_val)}')
print(f'  feature count  : {len(FEATURE_ORDER)}')
print(f'  has report json: {bool(report)}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — DATASET OVERVIEW
# ═══════════════════════════════════════════════════════════

print('Dataset Overview')
print('=' * 65)
print(f'Shape               : {df.shape}')
print(f'Unique tickers      : {df["ticker"].nunique() if "ticker" in df.columns else "N/A"}')
print(f'Date coverage       : {df["snapshot_date"].min() if "snapshot_date" in df.columns else "N/A"} -> {df["snapshot_date"].max() if "snapshot_date" in df.columns else "N/A"}')

if 'pseudo_label' in df.columns:
    print('\nLabel distribution (full):')
    print(df['pseudo_label'].value_counts(dropna=False).to_string())

if not df_train.empty and 'pseudo_label' in df_train.columns:
    print('\nLabel distribution (train):')
    print(df_train['pseudo_label'].value_counts(dropna=False).to_string())

if not df_val.empty and 'pseudo_label' in df_val.columns:
    print('\nLabel distribution (val):')
    print(df_val['pseudo_label'].value_counts(dropna=False).to_string())

print('\nSample rows:')
preview_cols = [c for c in ['ticker', 'snapshot_date', 'pseudo_label', 'rvol', 'rsi_14', 'extreme_language_ratio', 'source_diversity'] if c in df.columns]
print(df[preview_cols].head(10).to_string(index=False))

---
## Step 1: Data Collection
**Talking point:** *We pull 60 days of OHLCV data + 30 recent news headlines per ticker, with zero API keys for the core pipeline.*

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — DATA QUALITY CHECK
# ═══════════════════════════════════════════════════════════

required_cols = set(FEATURE_ORDER + ['pseudo_label'])
missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

na_counts = df[FEATURE_ORDER + ['pseudo_label']].isna().sum()
na_total = int(na_counts.sum())
print(f'Total missing values in core columns: {na_total}')

if na_total > 0:
    print('Top columns with missing values:')
    print(na_counts[na_counts > 0].sort_values(ascending=False).head(15).to_string())

before = len(df)
df = df.dropna(subset=['pseudo_label']).copy()
after = len(df)
print(f'Rows after pseudo_label cleanup: {after} (dropped {before - after})')

# Numeric guard
for col in FEATURE_ORDER:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

print('✓ Data quality checks passed')

---
## Step 2: Feature Engineering — Volume Features
**Talking point:** *RVOL (Relative Volume) is the core pump indicator. A 3x spike on GME with no earnings = red flag. The same spike on NVDA after an earnings beat = organic.*

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — EDA CHART A: FEATURE DISTRIBUTIONS BY LABEL
# ═══════════════════════════════════════════════════════════

KEY_FEATURES = [
    ('rvol',                   'Relative Volume (RVOL)',      'Higher = unusual trading interest'),
    ('volume_zscore',          'Volume Z-Score',              'Std dev above rolling mean'),
    ('rsi_14',                 'RSI (14)',                    '>75 often overbought'),
    ('extreme_language_ratio', 'Extreme Language Ratio',      'Higher = hype language density'),
    ('source_diversity',       'Source Diversity',            'Lower = concentrated narratives'),
    ('headline_similarity',    'Headline Similarity',         'Higher = repetitive narratives'),
]

features_present = [t for t in KEY_FEATURES if t[0] in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('EDA: Feature Distributions by Label', fontsize=13, y=1.01)

for ax, (feat, title, subtitle) in zip(axes.flat, features_present):
    labels_present = sorted(df['pseudo_label'].dropna().unique().tolist())
    for idx, label in enumerate(labels_present):
        subset = df[df['pseudo_label'] == label][feat].dropna()
        if len(subset) == 0:
            continue
        jitter = np.random.uniform(-0.08, 0.08, len(subset))
        x_pos = np.array([idx] * len(subset), dtype=float)
        color = LABEL_COLORS.get(label, GRAY)
        ax.scatter(x_pos + jitter, subset, color=color, alpha=0.6, s=18, edgecolors='none')
        ax.hlines(subset.mean(), idx - 0.18, idx + 0.18, color=color, linewidth=2.0)

    ax.set_xticks(range(len(labels_present)))
    ax.set_xticklabels(labels_present, rotation=30, ha='right', fontsize=9)
    ax.set_title(f'{title}\n{subtitle}', fontsize=10)
    ax.grid(axis='y', alpha=0.25)

# Hide any unused axes
for k in range(len(features_present), 6):
    axes.flat[k].axis('off')

handles = [mpatches.Patch(color=c, label=l) for l, c in LABEL_COLORS.items() if l in set(df['pseudo_label'])]
fig.legend(handles=handles, loc='lower center', ncol=max(2, len(handles)), bbox_to_anchor=(0.5, -0.02), fontsize=9)

plt.tight_layout()
plt.savefig('../data/eda_feature_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✓ Saved: ../data/eda_feature_distributions.png')

---
## Step 3: Feature Engineering — Price Features (RSI + Bollinger)
**Talking point:** *RSI > 75 + wide Bollinger Bands = overbought AND volatile. Alone, not enough to flag a pump. Combined with volume and news, it's part of the case.*

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — EDA CHART B: FEATURE VARIANCE
# ═══════════════════════════════════════════════════════════

feature_df = df[FEATURE_ORDER].copy()
variances = feature_df.var().sort_values(ascending=True)

LOW_VAR_THRESHOLD = 0.001
low_var_features = variances[variances < LOW_VAR_THRESHOLD].index.tolist()
good_features = variances[variances >= LOW_VAR_THRESHOLD].index.tolist()

bar_colors = [RED if v < LOW_VAR_THRESHOLD else BLUE for v in variances.values]

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(variances.index, variances.values, color=bar_colors, alpha=0.85)
ax.axvline(LOW_VAR_THRESHOLD, color=AMBER, linestyle='--', linewidth=1.3, label=f'Threshold={LOW_VAR_THRESHOLD}')
ax.set_xlabel('Variance')
ax.set_title('Feature Variance Across Expanded Dataset', fontsize=12)
ax.grid(axis='x', alpha=0.25)
ax.legend()

plt.tight_layout()
plt.savefig('../data/eda_feature_variance.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

print(f'Total features     : {len(FEATURE_ORDER)}')
print(f'Low-var (dropped)  : {low_var_features}')
print(f'Good features kept : {len(good_features)}')
print('✓ Saved: ../data/eda_feature_variance.png')

FEATURES_FINAL = [f for f in FEATURE_ORDER if f not in low_var_features]
print(f'\nFEATURES_FINAL ({len(FEATURES_FINAL)}): {FEATURES_FINAL}')

---
## Step 4: News Feature Engineering
**Talking point:** *We don't just ask 'is this positive news?' We ask: How many articles appeared in 6 hours? Are they all saying the same thing? Is the language extreme? These patterns are statistically different between organic moves and pumps.*

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — EDA CHART C: CORRELATION HEATMAP
# ═══════════════════════════════════════════════════════════

corr_cols = FEATURES_FINAL if 'FEATURES_FINAL' in globals() else FEATURE_ORDER
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8)

n = len(corr_matrix)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)
ax.set_title('Feature Correlation Matrix', fontsize=12)

if n <= 21:
    for i in range(n):
        for j in range(n):
            val = corr_matrix.iloc[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6, color='white' if abs(val) > 0.6 else '#888')

plt.tight_layout()
plt.savefig('../data/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

high_corr_pairs = []
for i in range(n):
    for j in range(i + 1, n):
        r = abs(corr_matrix.iloc[i, j])
        if r > 0.8:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r, 3)))

if high_corr_pairs:
    print('Highly correlated pairs (|r| > 0.8):')
    for a, b, r in high_corr_pairs[:25]:
        print(f'  {a:30s} ↔ {b:30s}  r={r}')
else:
    print('No highly correlated pairs above 0.8.')

print('✓ Saved: ../data/eda_correlation_heatmap.png')

---
## Step 5: Complete Feature Vector
**Talking point:** *Every raw data point goes through our pipeline and becomes a 21-dimensional feature vector. This is what the ML model will see — not raw prices, but engineered signals.*

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — TICKER-LEVEL PROFILE
# ═══════════════════════════════════════════════════════════

if 'ticker' not in df.columns:
    print('Ticker column not present; skipping ticker profile.')
else:
    grp_cols = [c for c in ['rvol', 'rsi_14', 'extreme_language_ratio', 'source_diversity'] if c in df.columns]
    ticker_profile = df.groupby('ticker')[grp_cols].mean().round(3)
    ticker_counts = df['ticker'].value_counts().rename('samples')
    profile = ticker_profile.join(ticker_counts, how='left').sort_values('samples', ascending=False)

    print('Top ticker sample counts:')
    print(profile.head(20).to_string())

    fig, ax = plt.subplots(figsize=(14, 6))
    ticker_counts.head(25).plot(kind='bar', ax=ax, color=BLUE, alpha=0.85)
    ax.set_title('Top 25 Tickers by Sample Count', fontsize=12)
    ax.set_ylabel('Rows')
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

---
## Step 6: Feature Correlation Heatmap
**Talking point:** *Before training any model, we check which features are correlated. Correlated features add noise, not signal. This heatmap guides our feature selection.*

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — TRAIN/VAL SPLIT INTEGRITY CHECK
# ═══════════════════════════════════════════════════════════

if df_train.empty or df_val.empty:
    print('Split files not found. Run Notebook 02 split cell to generate them.')
else:
    print('Split integrity check')
    print('─' * 65)
    print(f'train rows: {len(df_train)}')
    print(f'val rows  : {len(df_val)}')

    if 'pseudo_label' in df_train.columns and 'pseudo_label' in df_val.columns:
        print('\nTrain label distribution:')
        print(df_train['pseudo_label'].value_counts(dropna=False).to_string())
        print('\nVal label distribution:')
        print(df_val['pseudo_label'].value_counts(dropna=False).to_string())

    if report:
        print('\nLoaded training_report.json summary:')
        for key in ['rows_total', 'rows_train', 'rows_val', 'seed', 'test_size']:
            if key in report:
                print(f'  {key}: {report[key]}')
    else:
        print('\ntraining_report.json not found; continuing without it.')

---
## Step 7: The Hype Score — Summary
**Talking point:** *This is the core output of Phase 1. Before even training the ML model, our rule-based hype score already separates the three tickers clearly. The ML model will refine this further.*

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — NOTEBOOK 01 FINAL CHECKLIST
# ═══════════════════════════════════════════════════════════

artifacts = [
    '../data/training_data.csv',
    '../data/training_train.csv',
    '../data/training_val.csv',
    '../data/training_report.json',
    '../data/eda_feature_distributions.png',
    '../data/eda_feature_variance.png',
    '../data/eda_correlation_heatmap.png',
]

print('=' * 60)
print('NOTEBOOK 01 CHECKLIST')
print('=' * 60)

all_ok = True
for path in artifacts:
    ok = os.path.exists(path)
    icon = 'OK' if ok else 'MISSING'
    print(f'  [{icon:7s}] {path}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('Notebook 01 complete. Proceed to Notebook 02 for model training.')
else:
    print('Some artifacts are missing. Generate required files before moving on.')